In [ ]:
!pip install -q gensim transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 39.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import string

import nltk

from collections import Counter

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec
from transformers import AutoTokenizer, AutoModel

import matplotlib.pyplot as plt

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
import nltk
from nltk.corpus import brown
import pandas as pd

nltk.download('brown')
sentences = brown.sents()
texts = [' '.join(sentence) for sentence in sentences]

# Create DataFrame
df = pd.DataFrame({'text': texts})

df.head()
print("Number of sentences:", len(df))

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Number of sentences: 57340


In [ ]:
df.info();

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57340 entries, 0 to 57339
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    57340 non-null  object
dtypes: object(1)
memory usage: 448.1+ KB


In [ ]:
df.isnull().sum()

,0
text,0


In [ ]:
df['lower_text'] = df['text'].str.lower()
print(df['lower_text'].iloc[0])

the fulton county grand jury said friday an investigation of atlanta's recent primary election produced `` no evidence '' that any irregularities took place .


In [ ]:
import re

df['clean_text'] = df['lower_text'].apply(
    lambda x: re.sub(r'[^a-zA-Z\s]', '', x)
)

print(df['clean_text'].iloc[0])

the fulton county grand jury said friday an investigation of atlantas recent primary election produced  no evidence  that any irregularities took place 


In [ ]:
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

df['tokens'] = df['clean_text'].apply(word_tokenize)

print(df['tokens'].iloc[0])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'atlantas', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place']


In [ ]:
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

df['tokens_no_stopwords'] = df['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)

print(df['tokens_no_stopwords'].iloc[0])

['fulton', 'county', 'grand', 'jury', 'said', 'friday', 'investigation', 'atlantas', 'recent', 'primary', 'election', 'produced', 'evidence', 'irregularities', 'took', 'place']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()

df['stemmed_tokens'] = df['tokens_no_stopwords'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)

print(df['stemmed_tokens'].iloc[0])

['fulton', 'counti', 'grand', 'juri', 'said', 'friday', 'investig', 'atlanta', 'recent', 'primari', 'elect', 'produc', 'evid', 'irregular', 'took', 'place']


In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df['lemmatized_tokens'] = df['stemmed_tokens'].apply(
    lambda tokens: [lemmatizer.lemmatize(word) for word in tokens]
)

print(df['lemmatized_tokens'].iloc[0])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


['fulton', 'counti', 'grand', 'juri', 'said', 'friday', 'investig', 'atlanta', 'recent', 'primari', 'elect', 'produc', 'evid', 'irregular', 'took', 'place']


In [ ]:
tokenized_sentences = df['lemmatized_tokens'].tolist()

print("Number of sentences:", len(tokenized_sentences))
print("First processed sentence:", tokenized_sentences[0])

Number of sentences: 57340
First processed sentence: ['fulton', 'counti', 'grand', 'juri', 'said', 'friday', 'investig', 'atlanta', 'recent', 'primari', 'elect', 'produc', 'evid', 'irregular', 'took', 'place']
